# InfIndex Analysis
Inflammation index (InfIndex) and marker analysis for the PDP1 clinical trial.

In [ ]:
### Imports
import src.config as config
import sys
sys.path.append(config.path_szb_commons)
import commons_codebase.src.core as commons
import src.folders as folders
import src.core as core
import pandas as pd
import os

import warnings # We save whether model converged or not in output
import statsmodels
warnings.filterwarnings("ignore", category=statsmodels.tools.sm_exceptions.ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)

## Data Wrangling

In [ ]:
### Create master dataframe with all experiemntal data
# Exclude patient 1060 (not one of the 12 PDP1 completers - only has mitokine data at baseline, no other markers)
df_master = core.DataWrangl.get_df_master(
    exclude_pids=[1060],
    save={
        'dir_out': folders.exports, 
        'fname_out': 'infindex_master.csv'})
df_master.head(3)

In [ ]:
### Create table of observed means and SDs
df_master = pd.read_csv(os.path.join(folders.exports, 'infindex_master.csv'))
df_observed = commons.Analysis.get_df_observed(
    df_master=df_master.rename(columns={'value': 'score'})) 
    # get_df_observed() calculates the mean and SD of the 'score' column

# Sort rows
df_observed['measure'] = pd.Categorical(
    df_observed['measure'], 
    categories=config.markers, 
    ordered=True)
df_observed = df_observed.sort_values(by=['measure', 'tp']).reset_index(drop=True)

# Format measure names with Greek letters
df_observed['measure'] = df_observed['measure'].str.replace('TNF-alpha', 'TNF-α').str.replace('IFN-gamma', 'IFN-γ')
df_observed.to_csv(os.path.join(folders.exports, 'infindex_observed.csv'), index=False)
df_observed.head(3)

## Statistical models of change

In [ ]:
### Paired t-tests and Wilcoxon tests
df_pairedt_wilcoxon = core.Models.fit_paired_ttests(
    df_master=pd.read_csv(os.path.join(folders.exports, 'infindex_master.csv')),
    outcome='log_value',
    dir_out=folders.exports,
    fname_prefix='infindex_pairedt_wilcoxon')
df_pairedt_wilcoxon.head(3)

In [ ]:
### Mixed-effects models
df_mixedeffects = core.Models.fit_withinarm_septps(
    df_master=pd.read_csv(os.path.join(folders.exports, 'infindex_master.csv')), 
    outcome='log_value',
    dir_out=folders.exports,
    fname_prefix='infindex_mixedeffects')
df_mixedeffects.head(3)

# Correlations

In [ ]:
### Correlation matrices of change between biomarkers
markers = config.markers_infind_comps + config.markers_mitokines 
df_master = pd.read_csv(os.path.join(folders.exports, 'infindex_master.csv'))
df_master = df_master.rename(columns={'log_value': 'score'})

for tp in config.tps:
    df = commons.DataWrangl.widen_master(
        df_master=df_master,
        xvars=markers,
        x_tp=tp,
        x_use_delta=False,
        yvars=markers,
        y_tp=tp,
        y_use_delta=False)

    commons.Analysis.get_corrmats(
        df=df,
        xvars=markers,
        yvars=markers,
        draw=True,
        title=f'[1+ln(pg/mL)] correlation matrix at {tp}',
        dir_out=folders.corrmats,
        fname_out=f'infindex_corrmat_{tp}')

## Visualizations

In [10]:
### Combined (individual + group) concentration trajectories 
core.Plots.draw_conc_trajectory(
    df_master=pd.read_csv(os.path.join(folders.exports, 'infindex_master.csv')),
    outcome='log_value',
    draw_inds=True,
    draw_group=True,
    dir_out=folders.conc_trajectories,
    fname_prefix='infindex_trajectory_combined')

In [11]:
### Individual's concentration trajectories 
core.Plots.draw_conc_trajectory(
    df_master=pd.read_csv(os.path.join(folders.exports, 'infindex_master.csv')),
    outcome='log_value',
    draw_inds=True,
    draw_group=False,
    dir_out=folders.conc_trajectories,
    fname_prefix='infindex_trajectory_inds')

In [12]:
### Combined (individual + group) concentration trajectories with 95CI errorbars
core.Plots.draw_conc_trajectory_ci95(
    df_master=pd.read_csv(os.path.join(folders.exports, 'infindex_master.csv')),
    outcome='log_value',
    draw_inds=True,
    draw_group=True,
    dir_out=folders.conc_trajectories,
    fname_prefix='infindex_trajectory_ci95')

In [13]:
### Combined (individual + group) concentration trajectories with Morey (2008) within-subject error bars
# publication: https://www.tqmp.org/RegularArticles/vol04-2/p061/
core.Plots.draw_conc_trajectory_morey(
    df_master=pd.read_csv(os.path.join(folders.exports, 'infindex_master.csv')),
    outcome='log_value',
    draw_inds=True,
    draw_group=True,
    dir_out=folders.conc_trajectories,
    fname_prefix='infindex_trajectory_morey')

In [14]:
### Trajectories of mixed-effects model CIs
core.Plots.draw_conc_trajectory_mixedeffects(
    df_master=pd.read_csv(os.path.join(folders.exports, 'infindex_master.csv')),
    outcome='log_value',
    draw_inds=True,
    dir_out=folders.conc_trajectories,
    fname_prefix='infindex_trajectory_mixedeffects')

In [15]:
### Z-score heatmaps of change
core.Plots.draw_zscore_heatmap(
    df_master=pd.read_csv(os.path.join(folders.exports, 'infindex_master.csv')),
    dir_out=folders.heatmaps,
    fname_prefix='infindex_heatmap')

# Publication tables

In [17]:
# Inflamation markers table
df_mixed = pd.read_csv(os.path.join(folders.exports, 'infindex_mixedeffects_inflamation.csv'))
df_pairedt = pd.read_csv(os.path.join(folders.exports, 'infindex_pairedt_wilcoxon_inflamation.csv'))
df_pairedt = df_pairedt[['measure', 'tp', 'smd', 'smd_ci_low', 'smd_ci_high']]

df_mixed = pd.merge(df_mixed, df_pairedt, on=['measure', 'tp'], how='left')
df_mixed = df_mixed[[
    'measure', 'tp',
    'smd', 'smd_ci_low', 'smd_ci_high',
    'est', 'se', 'ci_low', 'ci_high', 
    'p', 'p_sig', 'p_adj', 'p_adj_sig']]

# Format measure names with Greek letters
df_mixed['measure'] = df_mixed['measure'].str.replace('TNF-alpha', 'TNF-α').str.replace('IFN-gamma', 'IFN-γ')
df_mixed.to_csv(os.path.join(folders.exports, 'infindex_Table1.csv'), index=False)
df_mixed.head(3)

,measure,tp,smd,smd_ci_low,smd_ci_high,est,se,ci_low,ci_high,p,p_sig,p_adj,p_adj_sig
0,infindex,A1,-0.72,-1.58,0.14,-0.464,0.143,-0.744,-0.184,0.001,**,0.003,**
1,infindex,B1,-0.60,-1.43,0.23,-0.408,0.165,-0.732,-0.084,0.014,*,0.021,*
2,infindex,B30,-0.62,-1.46,0.22,-0.392,0.197,-0.779,-0.005,0.047,*,0.047,*


In [19]:
# Exploratory markers table
df_mixed = pd.read_csv(os.path.join(folders.exports, 'infindex_mixedeffects_exploratory.csv'))
df_pairedt = pd.read_csv(os.path.join(folders.exports, 'infindex_pairedt_wilcoxon_exploratory.csv'))
df_pairedt = df_pairedt[['measure', 'tp', 'smd', 'smd_ci_low', 'smd_ci_high']]

df_mixed = pd.merge(df_mixed, df_pairedt, on=['measure', 'tp'], how='left')
df_mixed = df_mixed[[
    'measure', 'tp',
    'smd', 'smd_ci_low', 'smd_ci_high',
    'est', 'se', 'ci_low', 'ci_high', 
    'p', 'p_sig', 'p_adj', 'p_adj_sig']]

# Format measure names with Greek letters
df_mixed['measure'] = df_mixed['measure'].str.replace('TNF-alpha', 'TNF-α').str.replace('IFN-gamma', 'IFN-γ')

df_mixed.to_csv(os.path.join(folders.exports, 'infindex_Table1_exploratory.csv'), index=False)
df_mixed.head(3)

,measure,tp,smd,smd_ci_low,smd_ci_high,est,se,ci_low,ci_high,p,p_sig,p_adj,p_adj_sig
0,IL-7,A1,-0.75,-1.62,0.12,-0.277,0.129,-0.531,-0.023,0.032,*,0.252,NaN
1,IL-7,B1,0.59,-0.24,1.42,0.135,0.101,-0.062,0.333,0.180,NaN,0.509,NaN
2,IL-7,B30,-0.42,-1.22,0.38,-0.128,0.106,-0.336,0.079,0.225,NaN,0.534,NaN


In [21]:
# Mitokines table
df_mixed = pd.read_csv(os.path.join(folders.exports, 'infindex_mixedeffects_mitokines.csv'))
df_pairedt = pd.read_csv(os.path.join(folders.exports, 'infindex_pairedt_wilcoxon_mitokines.csv'))
df_pairedt = df_pairedt[['measure', 'tp', 'smd', 'smd_ci_low', 'smd_ci_high']]

df_mixed = pd.merge(df_mixed, df_pairedt, on=['measure', 'tp'], how='left')
df_mixed = df_mixed[[
    'measure', 'tp',
    'smd', 'smd_ci_low', 'smd_ci_high',
    'est', 'se', 'ci_low', 'ci_high', 
    'p', 'p_sig', 'p_adj', 'p_adj_sig']]

# Format measure names with Greek letters
df_mixed['measure'] = df_mixed['measure'].str.replace('TNF-alpha', 'TNF-α').str.replace('IFN-gamma', 'IFN-γ')

df_mixed.to_csv(os.path.join(folders.exports, 'infindex_Table1_mitokines.csv'), index=False)
df_mixed.head(3)

,measure,tp,smd,smd_ci_low,smd_ci_high,est,se,ci_low,ci_high,p,p_sig,p_adj,p_adj_sig
0,FGF-21,A1,0.41,-0.39,1.21,0.315,0.304,-0.281,0.910,0.300,NaN,0.360,NaN
1,FGF-21,B1,0.44,-0.37,1.25,0.431,0.243,-0.045,0.907,0.076,NaN,0.152,NaN
2,FGF-21,B30,-0.03,-0.80,0.74,-0.025,0.337,-0.685,0.635,0.941,NaN,0.941,NaN
